In [46]:
import torch
import os
from dotenv import load_dotenv

%env LDFLAGS=-L/opt/homebrew/opt/jpeg/lib
%env CPPFLAGS=-I/opt/homebrew/opt/jpeg/include
%env LDFLAGS="-L/opt/homebrew/opt/zlib/lib -L/opt/homebrew/opt/jpeg/lib"
%env CPPFLAGS="-I/opt/homebrew/opt/zlib/include -I/opt/homebrew/opt/jpeg/include"
%env PKG_CONFIG_PATH="/opt/homebrew/opt/zlib/lib/pkgconfig:/opt/homebrew/opt/jpeg/lib/pkgconfig"

!pip install pillow
!pip install llama-index
!pip install llama-cpp-python
!pip install -q -U google-genai  python-dotenv pandas
!pip install -q torch
!pip install jupyter llama-index-embeddings-huggingface
!pip install llama-cpp-python
!pip install -q python-dotenv 
!pip install -q -U google-genai

env: LDFLAGS=-L/opt/homebrew/opt/jpeg/lib
env: CPPFLAGS=-I/opt/homebrew/opt/jpeg/include
env: LDFLAGS="-L/opt/homebrew/opt/zlib/lib -L/opt/homebrew/opt/jpeg/lib"
env: CPPFLAGS="-I/opt/homebrew/opt/zlib/include -I/opt/homebrew/opt/jpeg/include"
env: PKG_CONFIG_PATH="/opt/homebrew/opt/zlib/lib/pkgconfig:/opt/homebrew/opt/jpeg/lib/pkgconfig"


In [49]:
from google import genai
from google.genai import types

GEMINI_API_KEY = ""

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing. Please set it in your environment or a .env file.")

# Initialize Gemini 1.5 Flash
llm_gemini = genai.Client(
    model="models/gemini-1.5-flash",
    api_key=GEMINI_API_KEY,
    temperature=0.1,
    max_tokens=2048
)



TypeError: Client.__init__() got an unexpected keyword argument 'model'

In [5]:
from llama_cpp import Llama
import os

# Download Mistral model if not already present
model_path = "content/mistral.gguf"
if not os.path.exists(model_path):
    !wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm = Llama(
        model_path=model_path,
        n_gpu_layers=1,  # Start with 1 layer on GPU to be safe
        n_ctx=2048,      # Context window size
        verbose=True     # Show loading progress
    )

    print("Model loaded successfully!")



except Exception as e:
    print(f"Error loading model: {e}")

ggml_metal_device_init: testing tensor API for f16 support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x105687cb0 | th_max = 1024 | th_width =   32


Model file exists. Size: 4166.07 MB


ggml_metal_device_init: testing tensor API for bfloat support
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'dummy_kernel', name = 'dummy_kernel'
ggml_metal_library_compile_pipeline: loaded dummy_kernel                                  0x105687cb0 | th_max = 1024 | th_width =   32
ggml_metal_library_init: using embedded metal library
ggml_metal_library_compile_all: compiled 'fa_aux' library in 0.003 sec
ggml_metal_library_compile_all: compiled 'fa_f16' library in 0.006 sec
ggml_metal_library_compile_all: compiled 'fa_f32' library in 0.029 sec
ggml_metal_library_compile_all: compiled 'fa_q4_0' library in 0.040 sec
ggml_metal_library_compile_all: compiled 'fa_q4_1' library in 0.028 sec
ggml_metal_library_compile_all: compiled 'fa_q5_0' library in 0.016 sec
ggml_metal_library_compile_all: compiled 'fa_q5_1' library in 0.038 sec
ggml_metal_library_compile_all: compiled 'fa_q8_0' library in 0.039 sec
ggml_metal_library_compile_all: compiled 'fa_vec_f16' library in 0.037 se

Model loaded successfully!


In [6]:
# Test with a RAG query
prompt = "What is RAG in the context of large language models?"
print(f"\nSending prompt: {prompt}")

response = llm(prompt, max_tokens=256, temperature=0.1)
print("\nResponse:")
print(response["choices"][0]["text"])


Sending prompt: What is RAG in the context of large language models?


ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_bin_fuse_f32_f32_f32_4', name = 'kernel_bin_fuse_f32_f32_f32_4_op=0_nf=1_rb=1_cb=0'
ggml_metal_library_compile_pipeline: loaded kernel_bin_fuse_f32_f32_f32_4_op=0_nf=1_rb=1_cb=0      0x10568bf30 | th_max = 1024 | th_width =   32
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_rms_norm_mul_f32_4', name = 'kernel_rms_norm_mul_f32_4'
ggml_metal_library_compile_pipeline: loaded kernel_rms_norm_mul_f32_4                     0x10568c9b0 | th_max = 1024 | th_width =   32
ggml_metal_library_compile_pipeline: compiling pipeline: base = 'kernel_mul_mv_q6_K_f32', name = 'kernel_mul_mv_q6_K_f32_nsg=2_ne12=1_r2=1_r3=1_split=0'
ggml_metal_library_compile_pipeline: loaded kernel_mul_mv_q6_K_f32_nsg=2_ne12=1_r2=1_r3=1_split=0      0x10568cf30 | th_max = 1024 | th_width =   32
llama_perf_context_print:        load time =    2573.05 ms
llama_perf_context_print: prompt eval time =    2572.14 ms /    13 tokens 


Response:


RAG, or Recursive Algebraic Graph, is a method used to efficiently represent and manipulate the complex dependencies and relationships between different components in large language models. It is a graph-based representation that allows for efficient computation of gradients during backpropagation, which is a crucial part of training these models.

In the context of large language models, RAG is used to represent the dependencies between different parts of the model, such as the self-attention mechanisms, feed-forward networks, and other components. By using RAG, researchers and developers can more easily understand the flow of information through the model, optimize its performance, and scale it up to handle larger datasets and more complex tasks.

RAG is particularly useful for models that have a large number of parameters and complex dependencies, such as transformer models, which are widely used in natural language processing and other areas of artificial intelligence 

## Integrating Open-Source LLMs into RAG (with a PDF!)

In [7]:
!pip install pymupdf
!pip install llama-index-llms-llama-cpp

In [8]:
!pip install llama-index-embeddings-huggingface

In [9]:
import fitz  # PyMuPDF

# Load the sample contract PDF
pdf_path = "sample-sdf-document.pdf"
doc = fitz.open(pdf_path)

# Extract text from all pages
text = "\n".join([page.get_text() for page in doc])

print(f"Extracted {len(text.split())} words from the PDF.")

Extracted 617 words from the PDF.


In [13]:
from llama_index.core import VectorStoreIndex, Document, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.settings import Settings
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure the LLM
llm = LlamaCPP(
    model_path="content/mistral.gguf",
    temperature=0.7,
    max_new_tokens=512,
    context_window=2048,
    model_kwargs={"n_gpu_layers": 1}
)

# Configure open-source embedding model
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"  # Lightweight but effective embedding model
)

# Set as the default LLM and embedding model
Settings.llm = llm
Settings.embed_model = embed_model

# Create documents from your text
documents = [Document(text=text)]  # 'text' should be your document content

# Build index
index = VectorStoreIndex.from_documents(documents)

# Configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=4,  # Retrieve 2 most similar chunks
)

# Configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)


llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.h

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [14]:
query = "What sterilization method was used for this product?"
query2 = "Summarize the key specifications in this document."
query3 =  "What are the storage conditions for this product?"

# Assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)


# Test querys

response = query_engine.query(query)
response2 = query_engine.query(query2)
response3 = query_engine.query(query3)
print("--------------------------------------------------------------")
print(query)
print("--------------------------------------------------------------")
print(response)
print("--------------------------------------------------------------")
print(query2)
print("--------------------------------------------------------------")
print(response2)
print("--------------------------------------------------------------")
print(query3)
print("--------------------------------------------------------------")
print(response3)

~llama_context:       MTL0 compute buffer size is  78.5000 MiB, matches expectation of  78.5000 MiB
~llama_context:        CPU compute buffer size is 176.0098 MiB, matches expectation of 176.0098 MiB
~llama_context:        CPU compute buffer size is   0.0000 MiB, matches expectation of   0.0000 MiB
ggml_metal_free: deallocating
llama_perf_context_print:        load time =   11311.76 ms
llama_perf_context_print: prompt eval time =   11309.07 ms /  1286 tokens (    8.79 ms per token,   113.71 tokens per second)
llama_perf_context_print:        eval time =    1252.65 ms /    31 runs   (   40.41 ms per token,    24.75 tokens per second)
llama_perf_context_print:       total time =   12567.14 ms /  1317 tokens
llama_perf_context_print:    graphs reused =         30
Llama.generate: 1270 prefix-match hit, remaining 16 prompt tokens to eval
llama_perf_context_print:        load time =   11311.76 ms
llama_perf_context_print: prompt eval time =     171.37 ms /    16 tokens (   10.71 ms per token

--------------------------------------------------------------
What sterilization method was used for this product?
--------------------------------------------------------------
 This product has been certified to conform to the autoclave and gamma irradiation sterilization methods as per the provided certificate of quality.
--------------------------------------------------------------
Summarize the key specifications in this document.
--------------------------------------------------------------

This document provides information about ÄKTA ready flow kits manufactured by Cytiva. The recommended storage temperature for standard ÄKTA ready flow kits is above +5°C, but they can be used at an operating temperature of +2°C to +40°C. Two specific modified flow kits, High Flow Kit F and High Flow Gradient C, are detailed, with their part numbers, operating temperatures, and access link to the operating instructions. The document also includes a certificate of quality, confirming the pro